## Random Forest Regression

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
df = pd.read_csv('cardekho_imputated.csv', index_col=[0])

In [3]:
# drop the unnecessary columns

df.drop(columns=['car_name', 'brand'], axis=1, inplace=True)

In [4]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']

In [5]:
# independant and dependant features

x = df.drop(columns=['selling_price'], axis=1)
y = df['selling_price']

In [6]:
# feature encoding and scaling

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [7]:
le = LabelEncoder()
x['model'] = le.fit_transform(x['model'])

In [8]:
one_hot_columns = ['transmission_type', 'seller_type', 'fuel_type']
num_features = x.select_dtypes(exclude='object').columns

In [9]:
one_hot = OneHotEncoder()
scaler = StandardScaler()

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('OneHotEncoder',one_hot, one_hot_columns),
        ('StandardScaler', scaler, num_features)
    ], remainder='passthrough'
)

In [11]:
x = preprocessor.fit_transform(x)

In [12]:
# trian test split

from sklearn.model_selection import train_test_split

In [13]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Model Training

In [17]:
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [16]:
# create a function to evaluate the model

def model_score(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    r2score = r2_score(true, predicted)

    return mae, mse, r2score

In [18]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'K-Neighbors': KNeighborsRegressor(),
    'AdaBoost Regressor': AdaBoostRegressor(),
    'Gradient Boosting': GradientBoostingRegressor()
}

In [19]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train, y_train)

    # make predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # evaluate the model
    model_train_mae, model_train_mse, model_train_r2_score = model_score(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_r2_score = model_score(y_test, y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance on Training Set')
    print("- MAE: {:.4f}".format(model_train_mae))
    print('- MSE: {:.4f}'.format(model_train_mse))
    print('- R2Score: {:.4f}'.format(model_train_r2_score))
    print('-'*30)
    print('Model Performance on Test Set')
    print("- MAE: {:.4f}".format(model_test_mae))
    print('- MSE: {:.4f}'.format(model_test_mse))
    print('- R2Score: {:.4f}'.format(model_test_r2_score))
    print('\n')

Linear Regression
Model Performance on Training Set
- MAE: 268101.6071
- MSE: 306756099359.7596
- R2Score: 0.6218
------------------------------
Model Performance on Test Set
- MAE: 279618.5794
- MSE: 252550062888.5655
- R2Score: 0.6645


Decision Tree Regressor
Model Performance on Training Set
- MAE: 5164.8199
- MSE: 432524990.5364
- R2Score: 0.9995
------------------------------
Model Performance on Test Set
- MAE: 124443.5804
- MSE: 92304858479.0473
- R2Score: 0.8774


Random Forest Regressor
Model Performance on Training Set
- MAE: 40146.4298
- MSE: 22228558817.5468
- R2Score: 0.9726
------------------------------
Model Performance on Test Set
- MAE: 101718.6379
- MSE: 51108075074.9995
- R2Score: 0.9321


Lasso
Model Performance on Training Set
- MAE: 268099.3635
- MSE: 306756104063.1791
- R2Score: 0.6218
------------------------------
Model Performance on Test Set
- MAE: 279614.9111
- MSE: 252549104707.3054
- R2Score: 0.6645


Ridge
Model Performance on Training Set
- MAE: 268060

In [ ]:
# hyperparameter tuning

rf_params = {
    "max_depth": [5, 8, 15, None, 10],
    "max_features": [5, 7, "auto", 8],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000]
}

gradient_params = {
    "loss": ['squared_error', 'huber', 'absolute_error'],
    "criterion": ['friedman_mse', 'squared_error', 'mse'],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, None],
    "max_depth": [5, 8, 15, None, 10],
    "learning_rate": [0.1, 0.01, 0.02, 0.03]
}

In [23]:
# models list

randomcv_models = [
    ('Random Forest', RandomForestRegressor(), rf_params),
    ('Gradient Bosoting', GradientBoostingRegressor(), gradient_params)
]

In [24]:
from sklearn.model_selection import RandomizedSearchCV

In [26]:
model_param = {}

for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model, param_distributions=params, n_iter=100, cv=3, verbose=2, n_jobs=-1)
    random.fit(x_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f'Best parameters for {model_name}:\n')
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits


d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
84 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\s

Fitting 3 folds for each of 100 candidates, totalling 300 fits


d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
138 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
34 fits failed with the following error:
Traceback (most recent call last):
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "d:\DS, ML, DL, NLP\Machine Learning\venv\Lib\

Best parameters for Random Forest:

{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 7, 'max_depth': 15}
Best parameters for Gradient Bosoting:

{'n_estimators': 500, 'min_samples_split': 8, 'max_depth': 15, 'loss': 'huber', 'learning_rate': 0.1, 'criterion': 'squared_error'}


In [27]:
tuned_models ={
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_split=2, max_features=7, max_depth=15, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=500, min_samples_split=8, max_depth=15, loss='huber', learning_rate=0.1, criterion='squared_error')
}

In [28]:
for i in range(len(list(tuned_models))):
    model = list(tuned_models.values())[i]
    model.fit(x_train, y_train)

    # make predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # evaluate the model
    model_train_mae, model_train_mse, model_train_r2_score = model_score(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_r2_score = model_score(y_test, y_test_pred)

    print(list(tuned_models.keys())[i])

    print('Model Performance on Training Set')
    print("- MAE: {:.4f}".format(model_train_mae))
    print('- MSE: {:.4f}'.format(model_train_mse))
    print('- R2Score: {:.4f}'.format(model_train_r2_score))
    print('-'*30)
    print('Model Performance on Test Set')
    print("- MAE: {:.4f}".format(model_test_mae))
    print('- MSE: {:.4f}'.format(model_test_mse))
    print('- R2Score: {:.4f}'.format(model_test_r2_score))
    print('\n')

Random Forest
Model Performance on Training Set
- MAE: 55011.4631
- MSE: 15395151053.8902
- R2Score: 0.9810
------------------------------
Model Performance on Test Set
- MAE: 97519.0252
- MSE: 44821901862.0550
- R2Score: 0.9405


Gradient Boosting
Model Performance on Training Set
- MAE: 6244.0668
- MSE: 525877254.9814
- R2Score: 0.9994
------------------------------
Model Performance on Test Set
- MAE: 104583.3270
- MSE: 50597455363.8406
- R2Score: 0.9328


